# Evaluación 2 · Primer análisis exploratorio de datos

**Módulo 1: Introducción y fundamentos estadísticos** · Diplomado en Ciencia de Datos Aplicada · UTFSM · INF074H

**Integrantes:** Daniela Contreras · Emanuel Nicklichen

**Proyecto:** *Quedar fuera teniendo puntaje* — revisión del proceso de Admisión a la Educación Superior 2026.

Objetivo de este notebook: aplicar a los datos del proyecto el checklist de calidad de la clase 2 — tipos, valores faltantes (incluidos los disfrazados), duplicados y valores imposibles según el dominio — antes de avanzar a estadística descriptiva y análisis bivariado. Cada decisión de limpieza se documenta en la celda donde se toma.

## 1. Los datos y su calidad

### 1.1 Carga desde la fuente declarada

Las cinco tablas provienen del portal de datos abiertos del Ministerio de Educación (declarado en la formulación), sección Educación Superior: `datosabiertos.mineduc.cl/pruebas-de-admision-a-la-educacion-superior/`. Se cargan desde `data/raw`, cada una en la carpeta que le corresponde.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DIRECTORIO_ACTUAL = Path.cwd()
DIRECTORIO_RAIZ = DIRECTORIO_ACTUAL.parent
RAW_DIR = DIRECTORIO_RAIZ / "data" / "raw"

# Carpetas esperadas dentro de data/raw, una por fuente (igual que en
# 00_verificacion_entorno.ipynb). Antes de leer nada, confirmamos que el
# archivo existe -- si no, un FileNotFoundError generico no dice cual falta.
carpetas = {
    "Inscritos":       "PAES-2026-Inscritos-Puntajes",
    "Socioeconomico":  "PAES-2026-Socioeconomicos",
    "Postulantes":      "PAES-2026-Postulantes",
    "Matricula":        "PAES-2026-Matricula",
    "Oferta":           "PAES-2026-Oferta-Definitiva-Programas",
}

for nombre, carpeta in carpetas.items():
    ruta = RAW_DIR / carpeta
    encontrados = list(ruta.glob("*.csv")) if ruta.exists() else []
    if encontrados:
        print(f"OK    - {nombre:15s} -> {encontrados[0].name}")
    else:
        print(f"ERROR - {nombre:15s} -> no se encontro ningun .csv en {ruta}")

Si alguna fila muestra `ERROR`, no seguir a la celda siguiente — revisar el nombre de la carpeta o si el `.csv` fue descomprimido en el lugar correcto. Confirmado que las 5 carpetas están OK, se cargan:

In [2]:
# Igual que arriba, pero usando glob() para tomar el .csv que encuentre en
# cada carpeta -- evita depender de escribir el nombre exacto del archivo,
# que es justo lo que rompio la primera version de esta celda.
bases = {}
for nombre, carpeta in carpetas.items():
    archivo = list((RAW_DIR / carpeta).glob("*.csv"))[0]
    if nombre == "Oferta":
        # Unica base con formato distinto: separador coma y comillas dobles
        bases[nombre] = pd.read_csv(archivo, sep=",", decimal=".")
    else:
        # sep=";" y decimal="," : formato confirmado en 01_exploracion_datos.ipynb
        bases[nombre] = pd.read_csv(archivo, sep=";", decimal=",", low_memory=False)

# Referencias cortas para el resto del notebook
A, B, C, D, O = bases["Inscritos"], bases["Socioeconomico"], bases["Postulantes"], bases["Matricula"], bases["Oferta"]

Con las cinco tablas cargadas, revisamos dimensiones y tipos en una sola pasada:

In [3]:
# Un ciclo for para no repetir el mismo bloque 5 veces: shape + info por tabla
for nombre, df in bases.items():
    print(f"{'=' * 10}")
    print(f"DATAFRAME: {nombre}")
    print(f"Dimensiones: {df.shape}")
    print()

DATAFRAME: Inscritos
Dimensiones: (320087, 131)

DATAFRAME: Socioeconomico
Dimensiones: (320087, 10)

DATAFRAME: Postulantes
Dimensiones: (1694081, 9)

DATAFRAME: Matricula
Dimensiones: (146873, 10)

DATAFRAME: Oferta
Dimensiones: (2150, 23)


Coincide exactamente con lo declarado en la formulación del proyecto (320.087 / 320.087 / 1.694.081 / 146.873 / 2.150). El tamaño combinado de las cinco tablas es consistente con los ~320 MB por proceso reportados en la estrategia de obtención.

### 1.2 Tipos de las variables relevantes

El proyecto no usa las 131+10+9+10+23 columnas disponibles, sino un subconjunto declarado en la formulación. Revisamos el tipo de cada una — este es el mismo paso que en `01_exploracion_datos.ipynb` reveló que `CLEC_REG_ACTUAL` llegaba como texto pese a ser numérica; hay que confirmarlo, no asumirlo, para cada variable nueva.

In [4]:
# Diccionario: base -> lista de variables que realmente usa el proyecto,
# segun la tabla "Variables relevantes" de la formulacion.
variables_por_base = {
    "Inscritos": ["MRUN","PTJE_NEM","PTJE_RANKING","CLEC_MAX","MATE1_MAX","MATE2_MAX",
                  "HCSOC_MAX","CIEN_MAX","HABILITACION_POST","DEPENDENCIA","RAMA_EDUCACIONAL"],
    "Socioeconomico": ["MRUN","INGRESO_PERCAPITA_GRUPO_FA"],
    "Postulantes": ["MRUN","ORDEN_PREF","ESTADO_PREF","PTJE_PREF","TIPO_PREF"],
    "Oferta": ["NEM","RANKING","CLEC","M1","HSCO","CIEN","M2","VAC_1ER"],
}

# Otro ciclo for: mismo chequeo (.dtypes) repetido para cada base, sin copiar y pegar
for nombre, cols in variables_por_base.items():
    print(f"--- {nombre} ---")
    print(bases[nombre][cols].dtypes)
    print()

--- Inscritos ---
MRUN                   int64
PTJE_NEM               int64
PTJE_RANKING           int64
CLEC_MAX               int64
MATE1_MAX              int64
MATE2_MAX              int64
HCSOC_MAX              int64
CIEN_MAX               int64
HABILITACION_POST      int64
DEPENDENCIA           object
RAMA_EDUCACIONAL      object
dtype: object

--- Socioeconomico ---
MRUN                            int64
INGRESO_PERCAPITA_GRUPO_FA      int64
dtype: object

--- Postulantes ---
MRUN              int64
ORDEN_PREF        int64
ESTADO_PREF       int64
PTJE_PREF        object
TIPO_PREF        object
dtype: object

--- Oferta ---
NEM        int64
RANKING    int64
CLEC       int64
M1         int64
HSCO      float64
CIEN      float64
M2        float64
VAC_1ER    int64
dtype: object


Dos cosas a marcar antes de seguir:

- **`PTJE_PREF` llega como texto** (`object`), igual que le pasó a `CLEC_REG_ACTUAL` en `01_exploracion_datos.ipynb`. Se revisa en la sección de valores disfrazados.
- **`HSCO`, `CIEN` y `M2` son `float64`** (no `int64` como el resto de Oferta) — es la primera pista de que tienen `NaN`: no todas las carreras ponderan estas tres pruebas, y pandas usa `float64` en vez de `int64` justamente porque hay ausencias reales en la columna.

### 1.3 Duplicados: fila completa y por llave

Se revisan en dos niveles, como corresponde: fila completa primero, y luego la llave que identifica a cada observación.

In [5]:
# Nivel 1: filas completas repetidas en cada tabla
for nombre, df in bases.items():
    print(f"{nombre:15s} {df.duplicated().sum()} filas duplicadas")

Inscritos       0 filas duplicadas
Socioeconomico  0 filas duplicadas
Postulantes     0 filas duplicadas
Matricula       0 filas duplicadas
Oferta          0 filas duplicadas


In [6]:
# Nivel 2: la llave que identifica a cada estudiante/observacion.
# En Inscritos, Socioeconomico y Matricula, MRUN deberia ser unico (una fila por estudiante).
print("MRUN duplicado en Inscritos:      ", A["MRUN"].duplicated().sum())
print("MRUN duplicado en Socioeconomico: ", B["MRUN"].duplicated().sum())
print("MRUN duplicado en Matricula:      ", D["MRUN"].duplicated().sum())

# En Postulantes, MRUN se repite por diseno (una fila por preferencia), asi que
# probamos primero con (MRUN, ORDEN_PREF), que deberia ser la llave logica.
print("\n(MRUN, ORDEN_PREF) duplicado en Postulantes:", C.duplicated(subset=["MRUN","ORDEN_PREF"]).sum())

MRUN duplicado en Inscritos:       0
MRUN duplicado en Socioeconomico:  0
MRUN duplicado en Matricula:       0

(MRUN, ORDEN_PREF) duplicado en Postulantes: 280569


`(MRUN, ORDEN_PREF)` da 280.569 duplicados — un número demasiado grande para ser error de carga. Antes de reportarlo como problema, se investiga un caso concreto:

In [7]:
# Aislamos un MRUN con la combinacion repetida para ver que esta pasando
dup_mask = C.duplicated(subset=["MRUN","ORDEN_PREF"], keep=False)
ejemplo = C.loc[dup_mask, "MRUN"].iloc[0]
print(C.loc[C["MRUN"] == ejemplo, ["MRUN","ORDEN_PREF","TIPO_PREF","COD_CARRERA_PREF"]].sort_values("ORDEN_PREF").to_string(index=False))

    MRUN  ORDEN_PREF TIPO_PREF  COD_CARRERA_PREF
      89           1   REGULAR             47001
      89           2   REGULAR             47002
      89           3   REGULAR             12025
      89           4   REGULAR             41037
      89           5   REGULAR             41082
      89           6   REGULAR             47491
      89           7   REGULAR             41068
      89           8   REGULAR             41072
      89           9   REGULAR             41136
      89          10   REGULAR             11522
      89          11   REGULAR             41002
      89          12   REGULAR             12006
      89          13   REGULAR             41031
      89          14   REGULAR             16109
      89          14    GENERO             16109
      89          15   REGULAR             41023

La preferencia N°14 aparece dos veces para el mismo `MRUN`: una vez como `REGULAR` y otra como `GENERO`, ambas apuntando a la misma carrera (`16109`). No es un duplicado real — un estudiante puede postular a la misma carrera por la vía regular y, en paralelo, por la vía de cupos de género, cada una con su propio proceso de selección. La llave que sí identifica una fila de forma única incluye la vía:

In [8]:
# Repetimos duplicated() agregando TIPO_PREF a la llave, para confirmar
# que ahora si es 0: cada (estudiante, posicion, via) debe ser unico.
print("(MRUN, ORDEN_PREF, TIPO_PREF) duplicado en Postulantes:", C.duplicated(subset=["MRUN","ORDEN_PREF","TIPO_PREF"]).sum())

(MRUN, ORDEN_PREF, TIPO_PREF) duplicado en Postulantes: 0


**Decisión documentada:** la llave única de `Postulantes` es `(MRUN, ORDEN_PREF, TIPO_PREF)`, no `(MRUN, ORDEN_PREF)`. Se usará esta llave en cualquier merge o verificación posterior sobre esta tabla.

### 1.4 Valores faltantes: reales y disfrazados

Un `NaN` es fácil de contar con `.isna()`. El problema son los valores que existen pero no son información real — hay que revisarlos variable por variable, porque un chequeo genérico no los detecta.

In [9]:
# Puntajes de Inscritos: en un dato int64 sin NaN, un valor de 0 sobre una escala
# de 100-1000 no puede ser un puntaje real -> es sospechoso de ser un centinela.
cols_puntaje = ["PTJE_NEM","PTJE_RANKING","CLEC_MAX","MATE1_MAX","MATE2_MAX","HCSOC_MAX","CIEN_MAX"]
for col in cols_puntaje:
    nulos = A[col].isna().sum()
    ceros = (A[col] == 0).sum()
    print(f"{col:15s} nulos reales: {nulos:>2}   en cero: {ceros:>7} ({ceros/len(A)*100:5.2f}%)")

PTJE_NEM        nulos reales:  0   en cero:    6129 ( 1.91%)
PTJE_RANKING    nulos reales:  0   en cero:    6129 ( 1.91%)
CLEC_MAX        nulos reales:  0   en cero:   43126 (13.47%)
MATE1_MAX       nulos reales:  0   en cero:   46817 (14.63%)
MATE2_MAX       nulos reales:  0   en cero:  193784 (60.54%)
HCSOC_MAX       nulos reales:  0   en cero:  123646 (38.63%)
CIEN_MAX        nulos reales:  0   en cero:  172002 (53.74%)


**Decisión documentada:** ninguna de las siete tiene `NaN`, pero todas tienen ceros que casi seguro son "no rindió" en vez de "puntaje 0" — más marcado en `MATE2_MAX`, `HCSOC_MAX` y `CIEN_MAX`, que son pruebas optativas (confirmado más abajo con el código `ESTADO_PREF = 35`, "no rindió ninguna de las pruebas opcionales"). El tratamiento de estos ceros queda pendiente para la etapa de modelado; aquí solo se documenta la magnitud.

In [10]:
# INGRESO_PERCAPITA_GRUPO_FA: el diccionario oficial define 1-10 como deciles
# y 99 como "Prefiere no responder" -- un valor valido para pandas, pero
# que para el analisis socioeconomico funciona como un dato faltante.
print(B["INGRESO_PERCAPITA_GRUPO_FA"].value_counts().sort_index())
n_99 = (B["INGRESO_PERCAPITA_GRUPO_FA"] == 99).sum()
print(f"\nCodigo 99 ('prefiere no responder'): {n_99} casos ({n_99/len(B)*100:.2f}%)")

INGRESO_PERCAPITA_GRUPO_FA
1     44641
2     33590
3     23658
4     21701
5     17318
6     15915
7     25266
8     20715
9     14455
10    14448
99    88380
Name: count, dtype: int64

Codigo 99 ('prefiere no responder'): 88380 casos (27.61%)

**Decisión documentada:** `INGRESO_PERCAPITA_GRUPO_FA = 99` se trata como faltante disfrazado, no como un decil real. Con 27,61% de los casos en esta categoría, cualquier análisis por decil socioeconómico (relevante para la hipótesis del proyecto) debe reportar este porcentaje como base excluida, no ignorarlo.

In [11]:
# PTJE_PREF llego como texto (visto en 1.2). Antes de asumir el motivo,
# probamos la conversion directa y aislamos lo que falla.
convertido = pd.to_numeric(C["PTJE_PREF"], errors="coerce")
fallidos = convertido.isna()
print("Filas que no convierten a numero:", fallidos.sum(), f"({fallidos.sum()/len(C)*100:.2f}%)")

# .unique() sobre los valores originales que fallaron, para ver que caracter es
print("Valor(es) que causan la falla:", C.loc[fallidos, "PTJE_PREF"].unique())

Filas que no convierten a numero: 131859 (7.78%)
Valor(es) que causan la falla: [' ']


El valor que rompe la conversión es un espacio en blanco `' '`, no texto real — el mismo patrón que muestra el material de la clase 2 (`' '` en vez de `NaN`). Antes de decidir qué hacer con esos 131.859 casos, se revisa si tienen una causa común:

In [12]:
# Cruzamos los PTJE_PREF en blanco con su ESTADO_PREF: si el patron es real,
# deberian concentrarse en causales de anulacion PREVIAS al calculo de puntaje.
print(C.loc[fallidos, "ESTADO_PREF"].value_counts().sort_index())

ESTADO_PREF
9      4188
15       16
17     7694
19      296
20      582
22       35
23       47
27     1433
28     1495
29     2674
30     5204
31     9394
35     7432
36    25156
40    15288
42    11084
51    39518
52      323
Name: count, dtype: int64

**Decisión documentada:** ninguno de estos códigos corresponde a `24` (seleccionado), `25` (espera) o `26` (seleccionado antes) — todos son causales de anulación **previas** al cálculo del puntaje ponderado (falta de NEM, código de carrera inválido, no cumple sexo exigido, prehabilitación PACE, etc., según el Anexo I del diccionario oficial). El espacio en blanco en `PTJE_PREF` no es un error: es la ausencia esperable de un puntaje que nunca llegó a calcularse. Se tratará como `NaN` real en los análisis siguientes, con esta causa documentada.

### 1.5 Valores imposibles según el dominio

In [13]:
# DEPENDENCIA: el diccionario declara codigos 1 a 6. Revisamos si existe algo mas.
print(A["DEPENDENCIA"].value_counts(dropna=False).sort_index())

DEPENDENCIA
       3200
1     19922
2     48543
3    168231
4     35248
5     10571
6     34372
Name: count, dtype: int64

**Decisión documentada:** 3.200 filas (1,00%) tienen `DEPENDENCIA` en blanco — un valor fuera del dominio declarado (1-6). Coincide en magnitud con otros vacíos vistos en `01_exploracion_datos.ipynb` para variables de egreso (región, comuna), donde la hipótesis del equipo fue estudiantes con estudios en el extranjero o situaciones de revalidación. No se confirma aquí con certeza — queda como valor a excluir de cualquier análisis por dependencia, documentando la pérdida (1,00% de la muestra).

In [14]:
# Ponderaciones de Oferta: cada programa deberia ponderar un total de 100%
# entre las 7 pruebas. Sumamos ingenuamente primero.
cols_pond = ["NEM","RANKING","CLEC","M1","HSCO","CIEN","M2"]
suma_ingenua = O[cols_pond].sum(axis=1)
print("Suma ingenua != 100:", (suma_ingenua != 100).sum(), "de", len(O), "programas")

Suma ingenua != 100: 1411 de 2150 programas


1.411 de 2.150 programas (65,6%) no suman 100 al sumar las 7 columnas directamente. Antes de reportar esto como un error masivo del dato, se revisó una ponderación real y vigente (Arquitectura, U. de Chile, admisión 2027): Historia y Ciencias no son dos pesos que se suman, son **un solo cupo de 10% que se llena con el mejor puntaje entre ambas pruebas** — el mismo criterio que "puntaje `_MAX`" aplica a nivel de ponderación.

In [15]:
# Corregimos: en vez de sumar HSCO + CIEN, usamos el maximo entre ambas
# (fillna(0) porque una carrera puede no pedir ninguna de las dos).
hsco_o_cien = O[["HSCO","CIEN"]].max(axis=1, skipna=True).fillna(0)
suma_corregida = O[["NEM","RANKING","CLEC","M1","M2"]].sum(axis=1) + hsco_o_cien

print("Suma corregida != 100:", (suma_corregida.round(2) != 100).sum(), "de", len(O), "programas")
print()
print("Carreras que persisten sin sumar 100:")
print(O.loc[suma_corregida.round(2) != 100, "CARRERA"].tolist())

Suma corregida != 100: 10 de 2150 programas

Carreras que persisten sin sumar 100:
['ACTUACIÓN TEATRAL', 'LICENCIADO/A EN ARTES CON MENCIÓN EN COMPOSICIÓN', 'DANZA', 'ACTUACIÓN', 'LICENCIATURA EN INTERPRETACIÓN MUSICAL', 'LICENCIATURA EN MÚSICA', 'ARTES MUSICALES Y SONORAS (VALDIVIA)', 'TEATRO', 'LICENCIATURA EN INTERPRETACIÓN Y FORMACIÓN MUSICAL ESPECIALIZADA', 'ACTUACIÓN Y CREACIÓN TEATRAL']


**Decisión documentada:** al usar el máximo entre `HSCO` y `CIEN` en vez de sumarlas, el problema baja de 1.411 a solo 10 programas (0,47%). Los 10 restantes son carreras artísticas (actuación, danza, música, teatro), que en la práctica exigen además una **prueba especial o audición** (confirmado contra un caso real: la ponderación oficial de Danza declara explícitamente una "Prueba Especial" que constituye el 50% del puntaje final) — un componente que esta base de Oferta no captura porque no forma parte del sistema PAES centralizado. No es un error del dato: es una limitación de cobertura, y se documenta como tal. Para el cálculo del puntaje ponderado del proyecto, se usará `MAX(HSCO, CIEN)` como ponderación conjunta, y se excluirán o marcarán aparte los 10 programas artísticos identificados.

### 1.6 Merges verificados: filas antes y después

In [16]:
# A + Socioeconomico: ambas tablas tienen 320.087 filas y comparten MRUN,
# deberian calzar 1 a 1 sin perdidas.
ab = A[["MRUN"]].merge(B[["MRUN","INGRESO_PERCAPITA_GRUPO_FA"]], on="MRUN", how="left")
print("Filas de A:", len(A), "| Filas tras merge A+B:", len(ab))
print("Filas de A sin match en B:", ab["INGRESO_PERCAPITA_GRUPO_FA"].isna().sum())

Filas de A: 320087 | Filas tras merge A+B: 320087
Filas de A sin match en B: 0


Sin pérdidas: A y B comparten exactamente la misma población de `MRUN`, como corresponde a dos vistas del mismo proceso de inscripción.

In [17]:
# A + Matricula: D es un subconjunto esperado de A (solo quienes matricularon).
ad = A[["MRUN"]].merge(D[["MRUN","NOMBRE_CARRERA"]], on="MRUN", how="left")
matricularon = ad["NOMBRE_CARRERA"].notna().sum()
print("Filas de A:", len(A), "| Filas tras merge A+D:", len(ad))
print("De A, matricularon:", matricularon, f"({matricularon/len(A)*100:.2f}%)")
print()

# Verificamos en el otro sentido: hay MRUN de D que no esten en A?
en_d_no_en_a = (~D["MRUN"].isin(A["MRUN"])).sum()
print("MRUN de D (matricula) que NO aparecen en A (inscritos):", en_d_no_en_a)
print("D total:", len(D), "= matricularon segun A (", matricularon, ") + no encontrados (", en_d_no_en_a, ")")

Filas de A: 320087 | Filas tras merge A+D: 320087
De A, matricularon: 141402 (44.18%)

MRUN de D (matricula) que NO aparecen en A (inscritos): 5471
D total: 146873 = matricularon segun A ( 141402 ) + no encontrados ( 5471 )


**Decisión documentada:** 5.471 matriculados (3,73% de `D`) no aparecen en la base de inscritos PAES en absoluto — no es un error de merge, sino evidencia de vías de ingreso que no requieren rendir la PAES del proceso vigente (por ejemplo, segunda carrera o admisión de titulados/graduados, mencionadas como hipótesis por el equipo). Estos 5.471 casos quedan fuera del alcance del proyecto, que se centra en el embudo de postulación PAES 2026; se documenta la exclusión, no se oculta.

### 1.7 Síntesis de calidad por tabla

In [18]:
# Tabla-resumen final: una fila por base, con lo encontrado en 1.3-1.6.
# No reemplaza el detalle de arriba, lo deja visible de un vistazo.
resumen_calidad = pd.DataFrame({
    "Tabla": ["Inscritos", "Socioeconomico", "Postulantes", "Matricula", "Oferta"],
    "Filas": [len(A), len(B), len(C), len(D), len(O)],
    "Duplicados (fila)": [0, 0, 0, 0, 0],
    "Faltantes disfrazados": [
        "CLEC/MATE1/MATE2/HCSOC/CIEN_MAX = 0 (centinela, no NaN)",
        "INGRESO_PERCAPITA_GRUPO_FA = 99 (27.61%)",
        "PTJE_PREF = ' ' (7.78%)",
        "Sin hallazgos nuevos (ver Inscritos para NOMBRE_REGION_EGRESO)",
        "Ponderaciones no capturan pruebas especiales (10 programas)",
    ],
    "Valor fuera de dominio": [
        "DEPENDENCIA en blanco (1.00%)",
        "-",
        "-",
        "-",
        "-",
    ],
})
resumen_calidad

,Tabla,Filas,Duplicados (fila),Faltantes disfrazados,Valor fuera de dominio
0,Inscritos,320087,0,"CLEC/MATE1/MATE2/HCSOC/CIEN_MAX = 0 (centinela, no NaN)",DEPENDENCIA en blanco (1.00%)
1,Socioeconomico,320087,0,INGRESO_PERCAPITA_GRUPO_FA = 99 (27.61%),-
2,Postulantes,1694081,0,PTJE_PREF = ' ' (7.78%),-
3,Matricula,146873,0,Sin hallazgos nuevos (ver Inscritos para NOMBRE_REGION_EGRESO),-
4,Oferta,2150,0,Ponderaciones no capturan pruebas especiales (10 programas),-


## Uso de Inteligencia Artificial

Se utilizó **Claude (Anthropic)** como asistente para la exploración de calidad de datos de este notebook: identificación y verificación de valores disfrazados, revisión de merges, y redacción de las interpretaciones. Se utilizó además **ChatGPT** para el código de visualización (histograma) originalmente generado en `01_exploracion_datos.ipynb`, reutilizado como base para los gráficos de notebooks previos del proyecto. El diseño de la pregunta de investigación, las decisiones metodológicas y la verificación final de cada hallazgo son del equipo; todas las cifras son reproducibles sobre las bases públicas declaradas.